# Notebook 08 - Cross-generator generalization (Section 5.3.4)

Evaluate best_vit (B1, trained on CIFAKE = Stable Diffusion v1.4) on images from
OTHER generators, following the GenImage cross-generator protocol.

**Data (you place it in Drive):**
```
luanvan_thac_si/genimage_ood/<Generator>/ai/      -> FAKE
luanvan_thac_si/genimage_ood/<Generator>/nature/  -> REAL
```
You do not need every generator - the notebook scans and runs whatever is present.

**Honest caveat (for the thesis):** GenImage real = ImageNet (large images), a
different domain from CIFAKE (32x32). So this is a cross-generator AND cross-domain
test; results are preliminary, on N images per class. No training here - inference
only. Every number is computed at run time.

In [15]:
# CELL 1: IMPORTS + DRIVE (force_remount picks up folders created after mounting)
import os, gc, random, numpy as np, pandas as pd, torch
from PIL import Image
from transformers import ViTForImageClassification, ViTImageProcessor
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from tqdm.auto import tqdm
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)  # refresh so new folders appear
except Exception as e:
    print('No Colab drive:', e)
print('Device:', DEVICE)

Mounted at /content/drive
Device: cpu


In [16]:
# CELL 2: CONFIG
ROOT = '/content/drive/MyDrive/luanvan_thac_si'
CONFIG = {
    'ood_dir'    : f'{ROOT}/genimage_ood',                  # <Gen>/ai , <Gen>/nature
    'best_vit'   : f'{ROOT}/notebook01_artifacts/best_vit', # B1 checkpoint from NB01
    'model_id'   : 'google/vit-base-patch16-224-in21k',     # for the image processor
    'n_per_class': 400,        # sample up to this many ai/ and nature/ per generator
    'eval_batch' : 64,
    'seed'       : SEED,
    'out_dir'    : f'{ROOT}/notebook08_artifacts',
}
os.makedirs(CONFIG['out_dir'], exist_ok=True)
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff')
print(CONFIG['ood_dir'])

/content/drive/MyDrive/luanvan_thac_si/genimage_ood


In [17]:
# CELL 3: *** STRUCTURE CHECK *** (recursive; handles <Gen>/train|val/ai|nature)
# GenImage nests images under train/ and val/. We search for FAKE/REAL class
# folders at ANY depth and prefer the val split when present.
FAKE_NAMES = {'ai', 'fake', 'fakes', '1_fake', 'ai_gen', 'synthetic'}
REAL_NAMES = {'nature', 'real', 'reals', '0_real', 'natural'}

def find_class_images(gen_dir, name_set):
    hits = []
    for dp, dns, fns in os.walk(gen_dir):
        if os.path.basename(dp).lower() in name_set:
            for f in fns:
                if f.lower().endswith(IMG_EXT):
                    hits.append(os.path.join(dp, f))
    val = [h for h in hits if (os.sep + 'val' + os.sep) in h.lower()]
    return val if val else hits          # prefer the val (test) split if available

def tree(d, prefix='   ', depth=2, _lvl=0):
    if _lvl >= depth: return
    try: entries = sorted(os.listdir(d))
    except Exception: return
    for e in entries[:8]:
        p = os.path.join(d, e)
        if os.path.isdir(p):
            print(prefix + e + '/'); tree(p, prefix + '   ', depth, _lvl + 1)

usable = []
if not os.path.isdir(CONFIG['ood_dir']):
    print('NOT FOUND:', CONFIG['ood_dir'])
    if os.path.isdir(ROOT):
        print('Mount sees in', ROOT, ':', sorted(os.listdir(ROOT)))
        print('If genimage_ood exists in Drive web UI but not here, the mount is stale:')
        print('Runtime > Disconnect and delete runtime, then Run all again.')
else:
    gens = sorted([g for g in os.listdir(CONFIG['ood_dir'])
                   if os.path.isdir(os.path.join(CONFIG['ood_dir'], g))])
    rows = []
    for g in gens:
        gp = os.path.join(CONFIG['ood_dir'], g)
        n_ai = len(find_class_images(gp, FAKE_NAMES))
        n_na = len(find_class_images(gp, REAL_NAMES))
        if n_ai == 0 or n_na == 0:
            note = 'MISSING a class -> skipped'
        elif min(n_ai, n_na) / max(n_ai, n_na) < 0.5:
            note = 'imbalance -> interpret with care'; usable.append(g)
        else:
            note = 'ok'; usable.append(g)
        rows.append({'generator': g, 'FAKE imgs': n_ai, 'REAL imgs': n_na, 'note': note})
    print(pd.DataFrame(rows).to_string(index=False) if rows else 'no generator subfolders.')
    print('\nUsable generators:', usable if usable else 'NONE')
    if not usable and gens:
        print('\nActual folder tree of first generator (to reveal the real layout):')
        print(gens[0] + '/'); tree(os.path.join(CONFIG['ood_dir'], gens[0]))
        print('\nExpected FAKE folder names:', sorted(FAKE_NAMES))
        print('Expected REAL folder names:', sorted(REAL_NAMES))


 generator  FAKE imgs  REAL imgs note
       ADM        400        400   ok
    BigGAN        400        400   ok
     GLIDE        400        400   ok
Midjourney        400        400   ok
      SDv5        400        400   ok
      VQDM        400        400   ok
    Wukong        400        400   ok

Usable generators: ['ADM', 'BigGAN', 'GLIDE', 'Midjourney', 'SDv5', 'VQDM', 'Wukong']


In [18]:
# CELL 4: LOAD best_vit (B1) + transform
model = ViTForImageClassification.from_pretrained(CONFIG['best_vit']).to(DEVICE).eval()
proc = ViTImageProcessor.from_pretrained(CONFIG['model_id'])
tf = Compose([Resize((224, 224)), ToTensor(),
              Normalize(mean=proc.image_mean, std=proc.image_std)])
id2label = {int(k): str(v).upper() for k, v in model.config.id2label.items()}
FAKE_IDX = next((i for i, n in id2label.items() if n == 'FAKE'), 0)  # NB01: FAKE=0
REAL_IDX = 1 - FAKE_IDX
print('id2label:', id2label, '| FAKE_IDX =', FAKE_IDX)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

id2label: {0: 'FAKE', 1: 'REAL'} | FAKE_IDX = 0


In [19]:
# CELL 5: HELPERS -- gather sampled paths (recursive), predict P(FAKE)
def gather(gen, n, rng):
    gp = os.path.join(CONFIG['ood_dir'], gen)
    fake = find_class_images(gp, FAKE_NAMES)
    real = find_class_images(gp, REAL_NAMES)
    rng.shuffle(fake); rng.shuffle(real)
    return [(p, True) for p in fake[:n]] + [(p, False) for p in real[:n]]

@torch.no_grad()
def predict(paths):
    pf = []
    for s in range(0, len(paths), CONFIG['eval_batch']):
        chunk = paths[s:s + CONFIG['eval_batch']]; imgs = []
        for p in chunk:
            try: imgs.append(tf(Image.open(p).convert('RGB')))
            except Exception: imgs.append(torch.zeros(3, 224, 224))
        logit = model(pixel_values=torch.stack(imgs).to(DEVICE)).logits
        pf.append(torch.softmax(logit.float(), 1).cpu().numpy()[:, FAKE_IDX])
    return np.concatenate(pf)

In [20]:
# CELL 6: EVALUATE PER GENERATOR
rng = random.Random(CONFIG['seed'])
results = []
for g in usable:
    items = gather(g, CONFIG['n_per_class'], rng)
    if not items:
        continue
    paths = [p for p, _ in items]
    gt_fake = np.array([f for _, f in items], dtype=int)   # 1 = FAKE, 0 = REAL
    pf = predict(paths)
    pred_fake = (pf >= 0.5).astype(int)
    acc = accuracy_score(gt_fake, pred_fake) * 100
    fake_recall = (pred_fake[gt_fake == 1] == 1).mean() * 100 if (gt_fake == 1).any() else float('nan')
    real_recall = (pred_fake[gt_fake == 0] == 0).mean() * 100 if (gt_fake == 0).any() else float('nan')
    try: auc = roc_auc_score(gt_fake, pf) * 100
    except Exception: auc = float('nan')
    f1 = f1_score(gt_fake, pred_fake) * 100
    results.append({'Generator': g, 'N': len(items), 'Accuracy (%)': round(acc, 2),
                    'AUC (%)': round(auc, 2), 'F1 (%)': round(f1, 2),
                    'FAKE recall (%)': round(fake_recall, 2), 'REAL recall (%)': round(real_recall, 2)})
    print(f"  {g:14s}: acc={acc:.2f}  auc={auc:.2f}  FAKErec={fake_recall:.1f}  REALrec={real_recall:.1f}")

df = pd.DataFrame(results)
if len(df):
    avg = {'Generator': 'AVERAGE', 'N': df['N'].sum(),
           'Accuracy (%)': round(df['Accuracy (%)'].mean(), 2),
           'AUC (%)': round(df['AUC (%)'].mean(), 2), 'F1 (%)': round(df['F1 (%)'].mean(), 2),
           'FAKE recall (%)': round(df['FAKE recall (%)'].mean(), 2),
           'REAL recall (%)': round(df['REAL recall (%)'].mean(), 2)}
    df = pd.concat([df, pd.DataFrame([avg])], ignore_index=True)
    print('\n' + df.to_string(index=False))
    df.to_csv(os.path.join(CONFIG['out_dir'], 'table_generalization.csv'), index=False)
    print('\nSaved', os.path.join(CONFIG['out_dir'], 'table_generalization.csv'))
else:
    print('No usable generator yet -- see CELL 3 output.')

  ADM           : acc=43.00  auc=40.85  FAKErec=51.7  REALrec=34.2
  BigGAN        : acc=35.88  auc=31.71  FAKErec=34.2  REALrec=37.5
  GLIDE         : acc=53.50  auc=59.77  FAKErec=71.5  REALrec=35.5
  Midjourney    : acc=61.88  auc=73.53  FAKErec=90.5  REALrec=33.2
  SDv5          : acc=60.88  auc=68.72  FAKErec=88.2  REALrec=33.5
  VQDM          : acc=50.25  auc=54.02  FAKErec=63.0  REALrec=37.5
  Wukong        : acc=59.38  auc=72.05  FAKErec=86.2  REALrec=32.5

 Generator    N  Accuracy (%)  AUC (%)  F1 (%)  FAKE recall (%)  REAL recall (%)
       ADM  800         43.00    40.85   47.59            51.75            34.25
    BigGAN  800         35.88    31.71   34.82            34.25            37.50
     GLIDE  800         53.50    59.77   60.59            71.50            35.50
Midjourney  800         61.88    73.53   70.36            90.50            33.25
      SDv5  800         60.88    68.72   69.28            88.25            33.50
      VQDM  800         50.25    54.02   55.

In [21]:
# CELL 7: READOUT -- how to interpret
print("How to read the results (Section 5.3.4):")
print(" - Accuracy ~50% and AUC ~50%  -> model does NOT generalize to that generator.")
print(" - Low FAKE recall (calls fake images real) -> ViT-SD fails to recognize the")
print("   artifacts of the other generator; a common, publishable negative finding.")
print(" - Low REAL recall (calls real images fake) -> domain shift (ImageNet vs CIFAKE).")
print(" - Literature anchor: a detector trained on one generator typically reaches")
print("   ~70% average cross-generator accuracy on GenImage. Compare against that.")
print("\nThesis caveat: this is a PRELIMINARY cross-generator AND cross-domain test")
print("(real = ImageNet), on N images per class; do not over-generalize beyond that.")

How to read the results (Section 5.3.4):
 - Accuracy ~50% and AUC ~50%  -> model does NOT generalize to that generator.
 - Low FAKE recall (calls fake images real) -> ViT-SD fails to recognize the
   artifacts of the other generator; a common, publishable negative finding.
 - Low REAL recall (calls real images fake) -> domain shift (ImageNet vs CIFAKE).
 - Literature anchor: a detector trained on one generator typically reaches
   ~70% average cross-generator accuracy on GenImage. Compare against that.

Thesis caveat: this is a PRELIMINARY cross-generator AND cross-domain test
(real = ImageNet), on N images per class; do not over-generalize beyond that.
